# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

## Step 1: Parse citations into (CITING, CITED) pairs
Filter out the header row, then split each remaining line on commas and 
strip quotes to get integer (CITING, CITED) tuples.

In [6]:
citations_header = rddCitations.first()
citations_data = rddCitations.filter(lambda line: line != citations_header)

def parse_citation(line):
    parts = line.replace('"', '').split(',')
    return (int(parts[0]), int(parts[1]))  # (CITING, CITED)

citationPairs = citations_data.map(parse_citation)
citationPairs.take(5)

[(3858241, 956203),
 (3858241, 1324234),
 (3858241, 3398406),
 (3858241, 3557384),
 (3858241, 3634889)]

## Step 2: Parse patents into (PATENT, STATE) pairs
Same approach: filter out the header row, then extract just the patent 
number and its state (POSTATE, column index 5). Missing states become 
None. Cached since this RDD is reused in multiple joins below.

In [7]:
patents_header = rddPatents.first()
patents_data = rddPatents.filter(lambda line: line != patents_header)

def parse_patent(line):
    parts = line.replace('"', '').split(',')
    patent = int(parts[0])
    state = parts[5] if parts[5] != '' else None   # POSTATE is column index 5
    return (patent, state)

patentState = patents_data.map(parse_patent)
patentState = patentState.cache()
patentState.take(5)

[(3070801, None),
 (3070802, 'TX'),
 (3070803, 'IL'),
 (3070804, 'OH'),
 (3070805, 'CA')]

## Step 3: Look up the cited patent's state
Re-key the citation pairs by CITED patent number, then join with 
patentState to attach the cited patent's state to each citation.

In [8]:
citedKeyed = citationPairs.map(lambda cc: (cc[1], cc[0]))   # (CITED, CITING)
citedJoined = citedKeyed.join(patentState)                  # (CITED, (CITING, CITED_STATE))
citedJoined.take(5)

[(3616380, (3858304, 'NJ')),
 (3616380, (3906540, 'NJ')),
 (3616380, (3927225, 'NJ')),
 (3616380, (3938243, 'NJ')),
 (3616380, (4034394, 'NJ'))]

## Step 4: Look up the citing patent's state
Re-key the result by CITING patent number and join with patentState 
again. Now every citation record has both the cited and citing patent's 
state. Cached since this is the expensive join, reused below.

In [9]:
citingKeyed = citedJoined.map(lambda kv: (kv[1][0], (kv[0], kv[1][1])))   # (CITING, (CITED, CITED_STATE))
bothJoined = citingKeyed.join(patentState)                                 # (CITING, ((CITED, CITED_STATE), CITING_STATE))
bothJoined = bothJoined.cache()
bothJoined.take(5)

[(5081667, ((3657720, 'NY'), 'CA')),
 (5081667, ((4893240, None), 'CA')),
 (5081667, ((4688244, 'CA'), 'CA')),
 (5081667, ((4809316, None), 'CA')),
 (5081667, ((3078834, 'UT'), 'CA'))]

## Step 5: Filter to same-state citations
Flatten the nested tuple structure and keep only citations where both 
states are known (non-null) and equal to each other -- these are the 
self-state citations.

In [10]:
same_state_rdd = bothJoined.map(
    lambda kv: (kv[0], kv[1][0][0], kv[1][0][1], kv[1][1])
    # (CITING, CITED, CITED_STATE, CITING_STATE)
).filter(
    lambda t: t[2] is not None and t[3] is not None and t[2] == t[3]
)

same_state_rdd.take(5)


[(5081667, 4688244, 'CA', 'CA'),
 (5081667, 4718080, 'CA', 'CA'),
 (5081667, 4887064, 'CA', 'CA'),
 (5081667, 4345554, 'CA', 'CA'),
 (5081667, 4750197, 'CA', 'CA')]

## Step 6: Count same-state citations per patent
Map each same-state citation to (CITING, 1) and reduce by key to get 
the total count per citing patent.

In [11]:
same_state_counts = same_state_rdd.map(lambda t: (t[0], 1)).reduceByKey(lambda a, b: a + b)
same_state_counts.take(5)

[(5081667, 6), (5967240, 1), (4238724, 1), (5630325, 4), (4991922, 3)]

## Step 7: Merge counts back onto the full patent list
Left outer join the counts onto patentState so every patent appears, 
filling in 0 for patents with no same-state citations.

In [12]:
augmented = patentState.leftOuterJoin(same_state_counts)  # (PATENT, (STATE, count_or_None))
augmented = augmented.mapValues(lambda v: (v[0], v[1] if v[1] is not None else 0))
augmented.take(5)

[(3070896, ('MT', 0)),
 (3070924, ('HI', 0)),
 (3071092, ('CA', 0)),
 (3071716, ('PA', 0)),
 (3072088, ('NY', 0))]

## Step 8: Sort and find the top 10
Sort all patents descending by same-state citation count and take the 
top 10. This matches the reference output (allowing for tie-breaking 
differences among patents with equal counts).

In [13]:
top10_rdd = augmented.map(lambda kv: (kv[0], kv[1][0], kv[1][1])) \
    .sortBy(lambda t: t[2], ascending=False) \
    .take(10)

for row in top10_rdd:
    print(row)
    

(5959466, 'CA', 125)
(5983822, 'TX', 103)
(6008204, 'CA', 100)
(5952345, 'CA', 98)
(5958954, 'CA', 96)
(5998655, 'CA', 96)
(5936426, 'CA', 94)
(5739256, 'CA', 90)
(5978329, 'CA', 90)
(5980517, 'CA', 90)
